# Harvesting under a strong Allee effect

<div style='border-left:5px solid #176b87;padding:12px;background:#176b8710'><b>Portable notebook</b><br>This file contains its Python source, configurations, synthetic datasets and recorded evidence. Run all cells from the top in a fresh kernel. No repository checkout or data download is needed.</div>

**Environment:** Python 3.11 or newer; install `numpy scipy matplotlib nbformat ipython`. The bootstrap restores the included source into a temporary directory. The readable source files are also included in the companion download.

**Scope:** WP01, WP02, WP03. All population examples are synthetic. A short demonstrator is distinct from recorded final experiments. Explanations, equations and algorithms precede the corresponding code; interpretations and limitations follow.

**Execution budget:** classical experiments are bounded by their saved configs. PINN demonstrations use the stated notebook settings; the full historical parameter sweeps are read from tables, not silently retrained.

In [ ]:
# Restore the source and synthetic evidence embedded in this notebook.
import base64
import hashlib
import io
import os
import sys
import tempfile
from pathlib import Path
from zipfile import ZipFile

_payload = base64.b64decode(
    "..."
)
assert (
    hashlib.sha256(_payload).hexdigest()
    == "1726986efbb6f6afdc5ef6141d5b29a3511972218d594cceb71ad7dcd8b33906"
)
ROOT = Path(tempfile.mkdtemp(prefix="mm-allee-notebook-"))
with ZipFile(io.BytesIO(_payload)) as _archive:
    for _entry in _archive.infolist():
        _path = (ROOT / _entry.filename).resolve()
        if not _path.is_relative_to(ROOT.resolve()):
            raise RuntimeError("Invalid embedded file path")
        _path.parent.mkdir(parents=True, exist_ok=True)
        _path.write_bytes(_archive.read(_entry))
sys.path.insert(0, str(ROOT / "src"))
os.environ["PYTHONPATH"] = str(ROOT / "src")
os.chdir(ROOT)
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    import matplotlib

    matplotlib.use("Agg")
print("Restored bundled source and synthetic data:", ROOT)

In [ ]:
import os
from pathlib import Path

import mm_allee

ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").exists() and (path / "src/mm_allee").exists()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the project checkout")
os.chdir(ROOT)
print("Package:", mm_allee.__version__)

## 1. Foundations and Model A: steady populations under a fixed catch

<div style="border-left:5px solid #2a78d6;background:#2a78d614;padding:10px 14px;border-radius:4px"><strong>Question answered in this section</strong><br>If we keep catching the same amount every year, at which population levels can the stock stay steady, and does it return to those levels after a small disturbance? Where is the largest catch that still allows a steady population?</div>

**Owner:** WP01. **Data type:** analytical derivations and numerical root finding/simulation of the stated ODEs; no field observations are used.
The same functions generate the report figures through `python experiments/wp01/run.py --config configs/wp01.json --output results/wp01`.

### 1.1 Model and assumptions

We model one closed, well-mixed population $N(t)\ge 0$ in continuous time with constant parameters $r>0$, $K>0$, $0<A<K$:

$$\text{Logistic: } \frac{dN}{dt}=rN\Big(1-\frac{N}{K}\Big)-H,\qquad
\text{Model A: } \frac{dN}{dt}=rN\Big(1-\frac{N}{K}\Big)\Big(\frac{N}{A}-1\Big)-H,\qquad
\text{Model B: } \frac{dN}{dt}=rN\Big(1-\frac{N}{K}\Big)\Big(\frac{N}{A}-1\Big)-hN.$$

- The factor $(N/A-1)$ is the **strong Allee effect**: growth is negative for $0<N<A$, so $A$ is a minimum viable population.
- $H$ (population/time) is a fixed quota; $h$ (1/time) is a fixed harvesting effort.
- Deterministic, unstructured, no migration; parameters constant.
- **Boundary convention:** a quota cannot be taken from an extinct stock, so when $N$ reaches $0$ the simulation stops (absorbing extinction).

<div style="border-left:5px solid #eb6834;background:#eb683414;padding:10px 14px;border-radius:4px"><strong>Why it matters</strong><br>A threshold population plus a fixed catch can make a stock collapse suddenly and fail to recover, the pattern seen in several marine fisheries.</div>

### 1.2 Nondimensionalisation

Set $x=N/K$, $\tau=rt$, $a=A/K\in(0,1)$, $q=H/(rK)$, $e=h/r$. The chain rule gives
$\dfrac{dN}{dt}=\dfrac{d(Kx)}{d\tau}\dfrac{d\tau}{dt}=rK\dfrac{dx}{d\tau}$, so dividing by $rK$:

$$\frac{dx}{d\tau}=G(x)-q,\qquad G(x)=x(1-x)\Big(\frac{x}{a}-1\Big).$$

Four dimensional parameters collapse to two shape parameters, $a$ and $q$. The cell below checks numerically that the dimensional right-hand side equals $rK$ times the dimensionless one.

In [ ]:
import numpy as np

from mm_allee.models import Parameters, dimensional_rhs, rhs

p = Parameters(
    r=0.5, K=1000.0, A=200.0, H=150.0
)  # illustrative values: 1/yr, tonnes, tonnes, tonnes/yr
d = p.dimensionless()
N = np.array([0.0, 100.0, 200.0, 500.0, 800.0, 1000.0])
lhs = dimensional_rhs(N, "A", p)
scaled = p.r * p.K * rhs(N / p.K, "A", a=d["a"], harvest=d["q"])
print("a = A/K =", d["a"], "  q = H/(rK) =", d["q"])
print("dN/dt (t/yr):       ", np.round(lhs, 6))
print("rK * dx/dtau (t/yr):", np.round(scaled, 6))
print("max difference:", np.max(np.abs(lhs - scaled)))

### 1.3 Equilibria and stability: the method

<div style="border-left:5px solid #1baf7a;background:#1baf7a14;padding:10px 14px;border-radius:4px"><strong>Definitions</strong><br>An <b>equilibrium</b> $x_e$ solves $f(x_e)=0$: the population stays put. <b>Stability</b> asks what a small disturbance does. In one dimension the sign of $f$ on each side decides: $f>0$ below and $f<0$ above means <i>stable</i>; the reverse means <i>unstable</i>; the same sign on both sides means <i>semistable</i> (attracts from one side only).</div>

When $f'(x_e)\neq0$, linearising $\xi=x-x_e$ gives $\dot\xi\approx f'(x_e)\xi$: stable if $f'<0$, unstable if $f'>0$, with relaxation time $1/|f'(x_e)|$.
When $f'(x_e)=0$ the linearisation says nothing, and only the one-sided signs decide.

### 1.4 Logistic baseline with constant yield

$f(x)=x(1-x)-q=0 \Rightarrow x_\pm=\tfrac12\big(1\pm\sqrt{1-4q}\big)$ and $f'(x_\pm)=1-2x_\pm=\mp\sqrt{1-4q}$.

| harvest | equilibria | stability |
|---|---|---|
| $q=0$ | $0,\ 1$ | $0$ unstable ($f'=1$), $1$ stable |
| $0<q<\tfrac14$ | $x_-<x_+$ | $x_-$ unstable (harvest-made threshold), $x_+$ stable |
| $q=\tfrac14$ | $x=\tfrac12$ (double) | semistable: $f=-(x-\tfrac12)^2\le0$ |
| $q>\tfrac14$ | none | every population collapses |

So the maximum constant yield is $H_c=rK/4$ at $N=K/2$.

In [ ]:
from mm_allee.equilibria import equilibria_logistic

for q in [0.0, 0.1, 0.25, 0.3]:
    recs = equilibria_logistic(q)
    text = (
        ", ".join(f"x={r.x:.4f} ({r.stability}, f'={r.derivative:+.3f})" for r in recs)
        or "none: collapse"
    )
    print(f"q={q:<5} -> {text}")

### 1.5 Model A: shape of the production curve and the fold

$$G(x)=-\frac{x^3}{a}+\frac{(1+a)x^2}{a}-x,\qquad G'(x)=-\frac{3x^2}{a}+\frac{2(1+a)x}{a}-1.$$

- Signs of the factors: $G<0$ on $(0,a)$, $G>0$ on $(a,1)$, $G<0$ for $x>1$, $G>0$ for $x<0$.
- $G'(x)=0 \iff 3x^2-2(1+a)x+a=0$. Since $G'(a)=1-a>0$ and $G'(1)=(a-1)/a<0$, the larger root lies in $(a,1)$ and is the maximum:

$$x^*=\frac{1+a+\sqrt{1-a+a^2}}{3},\qquad q_c=G(x^*)=\frac{2(1-a+a^2)^{3/2}+(1+a)(2a-1)(a-2)}{27a}.$$

Dimensionally $N^*=Kx^*$ and $H_c=rKq_c$. The cell checks the closed form against an **independent** numerical maximisation.

In [ ]:
from scipy.optimize import minimize_scalar

from mm_allee.equilibria import fold_a
from mm_allee.models import growth

for a in [0.1, 0.2, 0.3, 0.5]:
    x_star, q_c = fold_a(a)
    best = minimize_scalar(
        lambda x: -growth(x, a), bounds=(a, 1), method="bounded", options={"xatol": 1e-12}
    )
    print(
        f"a={a}: formula x*={x_star:.6f}, q_c={q_c:.8f} | numerical max x={best.x:.6f}, G={-best.fun:.8f}"
    )

### 1.6 Model A: the four harvest regimes

Equilibria solve $G(x)=q$. Because $G$ rises on $(a,x^*)$ and falls on $(x^*,1)$:

| regime | physical equilibria | stability |
|---|---|---|
| $q=0$ | $0,\ a,\ 1$ | stable, **unstable (minimum viable population)**, stable |
| $0<q<q_c$ | $x_-\in(a,x^*)$, $x_+\in(x^*,1)$ | $x_-$ unstable, $x_+$ stable; the third cubic root is **negative** and excluded |
| $q=q_c$ | $x^*$ (double root) | **semistable**: $f<0$ on both sides, attracts from above only |
| $q>q_c$ | none | collapse from every start |

<div style="border-left:5px solid #eb6834;background:#eb683414;padding:10px 14px;border-radius:4px"><strong>Zero is not an equilibrium when q > 0</strong><br>$f(0)=-q<0$. Below $x_-$ the population keeps falling with speed at least $q$ near zero and hits $x=0$ in finite time. We stop there (absorbing extinction); continuing the ODE would give negative populations.</div>

### 1.7 Algorithm for the roots

<div style="border-left:5px solid #6250d6;background:#6250d614;padding:10px 14px"><strong>Algorithm used by the integrated classical model</strong><br>1. Obtain the critical control from the shared threshold provider.<br>2. Only exact equality to that computed control denotes the double root. Nearby distinct controls remain below or above the fold.<br>3. Below the fold, bracket the two roots; close to it solve their small offsets from the maximum so populations near one remain distinguishable.<br>4. Always find the excluded negative root; classify the two positive branches using the factored cubic's signs, with independent high-precision and trajectory checks.<br>5. Report root residuals, derivatives and stability. If floating-point output cannot distinguish the roots, raise an explicit resolution error.</div>

The $64\varepsilon$ scale is a numerical uncertainty diagnostic, not a rule that creates an equilibrium above the fold. WP01 records and WP03 basin classification now use the same computed threshold and root provider. Initial-population uncertainty is separately labelled `boundary_unresolved`.

Why not a generic polynomial solver? Near the fold it can return a complex pair with a tiny imaginary part, as the next cell illustrates.

In [ ]:
from mm_allee.equilibria import equilibria_a

a = 0.2
x_star, q_c = fold_a(a)
print("numpy.roots at q = q_c:", np.roots([-1 / a, (1 + a) / a, -1.0, -q_c]))
for q in [0.0, 0.3, q_c - 1e-10, q_c, 0.63]:
    print(f"\nq = {q:.12f}")
    for r in equilibria_a(a, q, include_excluded=True):
        print(
            f"  x={r.x:+.8f}  {r.stability:<10} mult={r.multiplicity}  f'={r.derivative:+.3e}  "
            f"signs=({r.left_sign},{r.right_sign})  residual={r.residual:.1e}"
        )

### 1.8 Geometry, phase lines and branch diagram

Equilibria are where the production curve meets the horizontal harvest line. Filled circles are stable, open circles unstable, half-filled semistable.

In [ ]:
import matplotlib.pyplot as plt

from mm_allee.equilibria_plots import branch_figure, perturbation_figure, production_figure

fig = production_figure(a=0.2, logistic_q=(0.1, 0.25, 0.35), allee_q=(0.25 * q_c, q_c, 1.2 * q_c))
plt.show()

**Interpretation.** In the logistic panel the population only has a threshold when harvested. In Model A the threshold exists even at zero harvest ($x=a$), and the harvest line moves it up. At $q=q_c$ the line touches the top of the hill: one semistable point. Above it nothing can be sustained.

In [ ]:
fig = branch_figure(a=0.2)
plt.show()

**Interpretation.** The stable branch (solid) and the collapse threshold (dashed) approach each other as the quota rises and meet at the fold $(q_c,x^*)$, a saddle-node bifurcation. Implicit differentiation of $G(x)=q$ gives $dx/dq=1/G'(x)$: negative on the upper branch, positive on the lower one. The safety margin $x_+-x_-$ shrinks like $\sqrt{q_c-q}$.

### 1.9 Checking the stability labels by simulation

Independent check: start $\delta=0.01$ above and below each equilibrium and integrate with the shared RK45 solver (event at $x=0$).

In [ ]:
fig = perturbation_figure(a=0.2, q=0.3, delta=0.01, tau_end=12.0)
plt.show()

**Interpretation.** Both starts near $x_+$ return to it (stable). Just above $x_-$ the stock recovers to $x_+$; just below, it falls and reaches zero in finite time (unstable threshold). This is the behaviour the sign test predicted.

<div style="border-left:5px solid #1baf7a;background:#1baf7a14;padding:10px 14px;border-radius:4px"><strong>Main result</strong><br>For a fixed quota, the persistent state and the collapse threshold move together and merge at the maximum quota $q_c$. A shock causes collapse only if it exceeds the margin $x_+-x_-$, which shrinks to zero at $q_c$; harvesting near $q_c$ therefore leaves almost no safety margin and very slow recovery.</div>

<div style="border-left:5px solid #eb6834;background:#eb683414;padding:10px 14px;border-radius:4px"><strong>Limitations</strong><br>Deterministic and unstructured (no demographic noise, which matters most near the threshold); phenomenological cubic growth; constant parameters; the quota is assumed to be taken exactly whenever $N>0$. Dimensional numbers are illustrative only.</div>

## 2. Proportional harvesting and recovery

<div style="border-left:5px solid #c87116;background:#c8711614;padding:10px 14px"><strong>Question:</strong> Does reducing the catch restore a collapsed population? Carry state through the whole experiment and distinguish recovery from explicit restocking.</div>

Owner: @AshChan07. These are synthetic model calculations, not observations.

**Question:** does a smaller catch when population declines guarantee recovery?
No: below the Allee threshold, natural growth is already negative.
At 100 tonnes an effort of 0.05/year removes 5 tonnes/year, but at 20 tonnes
it removes 1 tonne/year. We compare this with a constant catch using common units.

## Derivation before computation
With $x=N/K$, $\tau=rt$, $a=A/K$, $e=h/r$,
$$f_B(x)=x[(1-x)(x/a-1)-e].$$
The nonzero roots solve $x^2-(1+a)x+a(1+e)=0$, so
$$x_\pm=\frac{1+a\pm\sqrt{(1-a)^2-4ae}}2,\qquad e_c=\frac{(1-a)^2}{4a}.$$
For $e<e_c$, $f_B=-x(x-x_-)(x-x_+)/a$ has signs $-,+,-$ on the three positive intervals.
Thus the lower positive root is unstable (the basin boundary) and the upper root is stable.
Zero is stable because $f'_B(0)=-1-e<0$. At the fold, the factor is a square:
the field is negative on both sides, so attraction occurs only from above.
Above the threshold only zero remains. At zero effort, recover $0,a,1$.

**Algorithm:** compute the discriminant threshold; return zero; return no positive
roots above it, one double root at it, or two roots below it. Use the product of
roots for the smaller root to avoid cancellation. The `e` field is effort, not
Task 1's yield `q`. Independent high-precision polynomial and trajectory tests
check this implementation.

In [ ]:
import csv
import json
import os
from pathlib import Path

from IPython.display import Image, display

from mm_allee.model_b import critical_effort, equilibria_b
from mm_allee.wp02_experiment import run_experiment

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
for effort in [0, 0.4, critical_effort(0.2), 1.0]:
    print(effort, [(r.x, r.stability) for r in equilibria_b(0.2, effort)])

## Fair comparison definitions

1. **Initial catch:** $H=hN_0$. Set $H=\alpha H_c^A$ and solve for $h$; the logistic model also gets $H$.
2. **Threshold fraction:** match $H_A/H_c^A=h/h_c=H_L/H_c^L=\alpha$. Initial catches can differ.

Use $r=0.5$/year, $K=1000$ tonnes, $A=200$ tonnes; $N_0/K=0.15,0.4,0.9$ and $\alpha=0.5,1.2$.
Calculate theoretical outcomes from phase lines separately from the finite observation horizon ($\tau=30$).
Integrate actual B catch over time; stop A/logistic catch at extinction. Divide total catch by the full horizon.
A blank critical population indicates no populated equilibrium. Equal $r,K$ does not match
logistic and Allee production curves, so a universal ecological risk ranking would be unjustified.

## History algorithm

Use staircase controls from 0 to 1.3 times critical and back, with 12 increments each way,
12 units at the peak, and dwell times 1 and 3 on ordinary intervals. Carry every terminal state forward.
After returning to zero, wait 15 units, then compare a 20-unit unassisted hold with a run adding $0.6K$.
For A, record the first zero event and shut down harvest; the zero continuation is a declared policy.
For B, never use the cutoff to reset the state. Report only the first sampled $x\leq10^{-6}$,
not an exact extinction time. Paired boundary rows encode left/right states; only restocking changes population discontinuously.

The following executes the actual bounded CPU experiment and writes a manifest. It also reruns
all four restocking histories with tighter integration settings; errors must be below the
predeclared absolute limits, $2\times10^{-7}$ in population and event time.

In [ ]:
config = json.loads((ROOT / "configs/wp02.json").read_text())
output = ROOT / "results/wp02-notebook"
manifest = run_experiment(config, output, ROOT)
print(manifest["status"], manifest["elapsed_seconds"], "seconds")
display(Image(filename=str(output / "wp02_b_branches.png")))
display(Image(filename=str(output / "wp02_history.png")))

In [ ]:
with (output / "wp02_comparison.csv").open() as f:
    comparison = list(csv.DictReader(f))
for row in comparison:
    if float(row["initial_population"]) == 900 and float(row["fraction"]) == 0.5:
        print(
            row["protocol"],
            row["model"],
            "initial catch",
            round(float(row["initial_catch"]), 2),
            "final biomass",
            round(float(row["final_population"]), 2),
            row["theoretical_outcome"],
        )
for row in manifest["summary"]:
    print(row)

## Interpretation and limits

At equal initial catch (131.28 tonnes/year), the example from 900 tonnes persists for A/B
but collapses for the logistic model: the catch exceeds that model's 125 tonnes/year maximum.
At half of each threshold, all three persist from 900 tonnes, but catches differ.
This contrast is about calibration, not evidence that Allee growth always improves safety.

For both ramp speeds, unassisted A reaches extinction and B becomes very small.
Reducing harvest to zero fails to restore a population below $a=0.2$.
Adding $0.6K$ at zero harvest crosses this threshold and restores the population toward $K$.
This is history dependence with an intervention, **not a spontaneous closed hysteresis loop**.
A event times in the curated run are about 13.436601 and 34.612578 in $\tau$ units;
B's first sampled smallness times are 20.4 and 42.0. Dynamic delay is not a new equilibrium threshold.
The two schedules also change total exposure, not just ramp speed.

Refinement errors in the curated runs are below $2\times10^{-9}$; tiny B tails below solver
absolute tolerance have no claimed relative accuracy. Test checks include high-precision roots,
independent maximisation, perturbations, matched units, boundary continuity, insufficient and sufficient
restocking, numerical refinement and manifest checksums. See `docs/tasks/WP02_model_b.md` for the evidence map.

The integrated notebook uses current reviewed WP01 code and WP03's shared threshold/basin helpers.
Comparisons retain the validated range $0.05\leq a\leq0.95$ and conservatively exclude ambiguous near-fold inputs.
A near-boundary initial population is explicitly unresolved. Final group slides, human reviews and rehearsal remain pending.
Optional slowing-down experiments are deferred. Codex assisted drafting and implementation;
human understanding has not been certified.

**Presentation check:** why does zero remain stable when effort falls back to zero?
Explain the sign of natural growth below the Allee threshold before quoting a formula.

## 3. Collapse thresholds, finite-time extinction and basins
<div style="border-left:4px solid #16803d;padding:10px"><strong>WP03 analysis implemented.</strong> Synthetic deterministic models; final group integration, human reviews and Devam's detailed learning review remain pending.</div>

For a well-mixed population, assume constant \(r>0\), \(0<A<K\), nonnegative harvest, no delays/age structure or other species. Put \(x=N/K,\ \tau=rt,\ a=A/K,\ q=H/(rK),\ e=h/r\). Then
\[
x'=G(x)-q\quad(A),\qquad x'=G(x)-ex\quad(B),\qquad
G(x)=-x^3/a+(1+a)x^2/a-x.
\]
**Threshold derivation.** \(G'=(-3x^2+2(1+a)x-a)/a\). The larger stationary point is
\(x_*=(1+a+\sqrt{1-a+a^2})/3\), in \((a,1)\), with \(G''(x_*)<0\);
the smaller point lies below a. Thus \(q_c=G(x_*)\) and
\(N_*=(K+A+\sqrt{K^2-KA+A^2})/3,\ H_c=rKq_c\).
For B, maximise \(P(x)=G(x)/x=(1-x)(x/a-1)\):
\(P'=(1+a-2x)/a,\ P''=-2/a\), so
\(e_c=(1-a)^2/(4a)\), \(h_c=re_c\).
The logistic class maximum is \(1/4\), giving \(H_{c,\mathrm{logistic}}=rK/4\).
H and h have different units; threshold comparisons also depend on production calibration.

**Algorithm:** compute closed-form thresholds; check them against independent bounded numerical maxima of expanded polynomials; classify basins before any extinction integration.

In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

from mm_allee.models import Parameters
from mm_allee.thresholds import dimensional_thresholds

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "configs/wp03.json").exists() and (p / "src/mm_allee").exists()
)
config = json.loads((ROOT / "configs/wp03.json").read_text())
parameters = Parameters(**config["parameters"])
print("Synthetic parameters:", vars(parameters))
print("Dimensional thresholds:", dimensional_thresholds(parameters))

### Basins and critical cases
For \(0<q<q_c\), A has two positive roots \(x_-<x_+\) and signs \(-,+,-\): below the lower root extinction, above it approach to the upper root. Exact roots remain equilibria. For B,
\[
x_\pm=(1+a\pm\sqrt{(1-a)^2-4ae})/2.
\]
At positive roots \(f'_B=xP'\), so the lower is unstable and upper stable; zero has derivative \(-(1+e)<0\).

| Control | A | B |
|---|---|---|
| Zero | Below a approaches zero asymptotically; above a tends to 1 | Same positive-state basins |
| Below fold | Below lower root reaches zero; above tends to upper root | Below lower root approaches zero; above tends to upper root |
| At fold | Below double root reaches zero; above approaches it | Below double root approaches zero; above approaches it |
| Above fold | Every finite positive start reaches zero | Every positive start approaches zero |

Exact initial equilibria stay there in all rows. At a fold the derivative is zero but signs on both sides are negative: **semistable**, attracting only from above.
The classifier explicitly recognises returned roots and marks non-exact starts within \(2\,10^{-12}\) of an unstable/semistable boundary unresolved. It does not merge adjacent harvest controls into the fold or decide survival from a finite horizon.

Numerically, compute the B upper root first and use the quadratic root product `lower = a*(1+e)/upper` to avoid cancellation for small a; the exact fold remains an explicit repeated root. The integrated WP01–WP03 APIs use the same threshold and branch providers and exact equality to the computed critical control. Adjacent controls retain distinct topology; numerical and initial-state uncertainty are reported separately.


### Extinction integral and solver checks
On a reaching-zero A path,
\[
\tau_{\rm ext}=\int_0^{x_0}\frac{du}{q-G(u)},\qquad
t_{\rm ext}=\tau_{\rm ext}/r=\int_0^{N_0}\frac{dN}{H-g(N)}.
\]
For a subcritical start below the barrier the denominator is continuous and positive over the whole path; at/above an intervening equilibrium the extinction integral is inapplicable. For \(q>q_c\), \(G\le q_c\) gives \(\tau_{\rm ext}\le x_0/(q-q_c)\).
For zero-yield A and B, smoothness/uniqueness exclude a positive trajectory reaching zero in finite time.

**Algorithm:** classify -> integrate only reaching-zero paths -> compare quadrature with two RK45 tolerances and independently expanded-polynomial DOP853 -> retain every case/status. No equilibrium pole is crossed. Blank times mean no finite extinction or unresolved boundary, not zero.

RK45 uses a shared-stage 5(4) pair for adaptive error control. We take the fifth-order state, locate the first downward zero event for positive-yield A, and include its endpoint without adding a zero tail. Output times are separate from adaptive internal steps. Quadrature error estimates and local RK tolerances are not rigorous global error bounds. References: [SciPy solve_ivp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html), [quad](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.quad.html).

**Difficult endpoint handling:** near the subcritical barrier, a logarithmic variable removes the vanishing linear factor. At the computed fold an elementary partial-fraction expression evaluates the extinction integral (`fold_closed_form`, no quadrature-error estimate). This avoids missed endpoint contributions for close-below-fold starts. Root rounding is separate from integration error. See `docs/WP03_EXPLAINED.md` for a basics-first explanation and the derivation.

Near a subcritical boundary, \(d\tau_{\rm ext}/dx_0 = 1/[q-G(x_0)] \sim 1/[f_A'(x_-)(x_--x_0)]\). Rounding in the initial state or root is amplified as the gap shrinks; small integration error alone does not justify extra digits.


In [ ]:
outputs = {
    "analysis": ROOT / "results/wp03-notebook",
    "reference": ROOT / "results/wp03-reference-notebook",
}
commands = [
    ("analysis", "experiments/wp03/run.py", "configs/wp03.json"),
    ("reference", "experiments/wp03/generate_reference.py", "configs/wp03_reference.json"),
]
for label, script, configuration in commands:
    completed = subprocess.run(
        [
            sys.executable,
            str(ROOT / script),
            "--config",
            str(ROOT / configuration),
            "--output",
            str(outputs[label]),
        ],
        cwd=ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    print(completed.stdout.strip())


def read_rows(filename):
    with (outputs["analysis"] / filename).open(newline="") as stream:
        return list(csv.DictReader(stream))


evidence = read_rows("wp03_event_validation.csv")
assert all(row["status"] == "passed" for row in evidence)
print(
    "Largest recorded event difference in dimensionless time:",
    max(float(row["max_abs_error_tau"]) for row in evidence),
)
display(Image(filename=str(outputs["analysis"] / "wp03_basins.png")))
display(Image(filename=str(outputs["analysis"] / "wp03_extinction.png")))

### Interpret the synthetic curves
The default \(r=0.7,K=100,A=20\) gives \(N_*=70.55050\), \(H_c=36.75966\), \(h_c=0.56\); logistic gives \(17.5\).
The 427 extinction evaluations retain non-finite outcomes rather than turning them into zero. Initial zero is omitted on the logarithmic time axis. Solid basin boundaries mark critical initial populations; the finite colour grid does not replace exact equilibrium classification.

At fixed reaching-zero initial population, increasing catch reduces extinction time. Approaching a basin boundary can make that time very large. A finite simulation horizon without an event does not settle whether extinction happens later. The six event checks include the fold from below and supercritical passage; the observed differences are much smaller than the declared \(2\,10^{-7}+5\,10^{-7}\tau_{\rm ext}\) acceptance tolerance.

Basin colour regions meet exactly at control/critical = 1; the fold is drawn as a line, so persistence cells do not extend above it. Every WP03 figure is exported as PNG/EPS/vector PDF; extensionless report references select the PDF under pdflatex.


### Why collapse can look delayed: the bottleneck
With \(q=q_c+\delta,\ y=x-x_*\), the exact cubic expansion is
\[
q-G(x)=\delta+\beta y^2+y^3/a,\qquad \beta=\sqrt{1-a+a^2}/a.
\]
For a fixed start above \(x_*\), the trajectory crosses a slow region. Integrating the leading denominator gives
\[
\int\frac{dy}{\delta+\beta y^2}
=\frac{\arctan(y\sqrt{\beta/\delta})}{\sqrt{\beta\delta}},
\quad \tau_{\rm ext}\sim\pi/\sqrt{\beta\delta}.
\]
The full near-fold integral is evaluated with \(y=\sqrt{\delta/\beta}\tan\theta\) to resolve its narrow peak.
This is passage after the equilibria disappear, distinct from relaxation to a stable equilibrium. A start already below the bottleneck need not have this divergence. At the fold, approach from above is algebraic, \(y\simeq y_0/(1+\beta y_0\tau)\).

In [ ]:
rows = read_rows("wp03_bottleneck.csv")
for row in rows:
    print("delta/qc =", row["fraction_above_fold"], "| tau*sqrt(delta) =", row["scaled_time"])
print("Leading scaled-time limit:", rows[0]["asymptotic_scaled_time"])
display(Image(filename=str(outputs["analysis"] / "wp03_bottleneck.png")))

### Optional results: sensitivity and sustainable yield
Holding other dimensional r,K,A fixed, \(g_N(N_*)=0\) eliminates the derivative of \(N_*\) from the maximum:
\[
H_{c,r}=H_c/r,\quad H_{c,K}=rN_*^2(N_*/A-1)/K^2,\quad
H_{c,A}=-rN_*^2(1-N_*/K)/A^2.
\]
Also
\[
h_{c,r}=h_c/r,\quad h_{c,K}=\tfrac r4(1/A-A/K^2),\quad
h_{c,A}=\tfrac r4(-K/A^2+1/K).
\]
Central differences check these derivatives; elasticities are \((p/C)C_p\). An a sweep at fixed r,K is a different experiment.

A's stable-branch yield supremum \(H_c\) lies at the semistable cliff, with no control margin.
For B, \(Y(h)=hN_+(h)=g(N_+(h))\). The production maximiser is on the upper branch, so
\(h_{\rm MSY}=g(N_*)/N_*<h_c,\ Y_{\rm MSY}=H_c\).
Independent bounded numerical maximisation checks the optimum. The example effort margin at MSY is about 6.96%; the initial state must still exceed the unstable root. Separate illustrative A/B policies are not a universal risk ranking.

In [ ]:
summary = json.loads((outputs["analysis"] / "wp03_summary.json").read_text())
print(json.dumps(summary, indent=2))
display(Image(filename=str(outputs["analysis"] / "wp03_sensitivity_yield.png")))
display(Image(filename=str(outputs["reference"] / "wp03_rk45_reference.png")))

### Handoff, reproduction and limitations
The early handoff retains six clean A/B CSVs (606 points). WP05 owns noise, frozen splits, inverse PINN/ODE fitting and held-out validation; WP04 supplies the shared neural engine. Keep known conditions/fixed external scales separate from evaluation truth, and never fit against held-out clean values or hidden unknown parameters.

Exact commands and test/evidence mapping: **docs/WP03_ANALYSIS.md**. The experiment regenerates tables and PNG/EPS; manifests record versions, units, tolerances and source/file hashes. All data are synthetic; no empirical species-level conclusion follows.

The report, notebook and EPS files required by the course must ultimately be integrated with all five tasks. **scripts/check_submission_ready.py** aggregates missing final artifacts, unfinished sections and metadata; a successful WP03 run does not make the final group submission ready. Reviews and non-author reproduction remain required.

AI assistance: OpenAI ChatGPT/Codex for this implementation, mathematical writing, tests and documentation, recorded in docs/AI_USAGE.md. Devam's detailed understanding review remains pending as requested.